# RAG de primeros auxilios (versión mejorada)

Este notebook parte de la misma idea del `RAG_simple` del profesor (embeddings + similitud coseno + generación), pero lo adapta al problema del proyecto: un manual largo, en español, donde cada respuesta debe poder rastrearse hasta su **documento y página**.

**Qué cambia frente al ejemplo del profesor**

| Pieza | `RAG_simple` | Esta versión |
|---|---|---|
| Base de conocimiento | 7 frases sueltas | PDF(s) real(es), dividido(s) en fragmentos con metadatos |
| Embeddings | `all-MiniLM-L6-v2` (inglés) | modelo multilingüe |
| Recuperación | 1 pasaje, solo semántica | top-k, semántica + palabras clave (híbrida), expansión de consulta opcional |
| "No sé" | no existe | umbral de similitud |
| Fuentes | no | documento + página, armadas por el código (no por el modelo) |
| Evaluación | no | preguntas de prueba con páginas esperadas y comparación entre versiones |

**Orden de ejecución:** de arriba hacia abajo. Si no tienes GPU, pon `USAR_GENERADOR = False` en la configuración y el sistema mostrará los fragmentos recuperados en vez de redactar una respuesta.

In [ ]:
# 1. Instalación (Colab). Quita el '#' de la línea de OCR solo si el PDF sale con texto ilegible.
!pip install -q pymupdf sentence-transformers transformers accelerate scikit-learn pandas
# !apt-get -qq install -y tesseract-ocr tesseract-ocr-spa && pip install -q pytesseract pillow

## 1. Configuración

Aquí está todo lo que el grupo puede ajustar sin tocar el resto del código. Para agregar más fuentes (ICBF, CAR, etc.) basta con añadir otro diccionario a `DOCUMENTOS`.

`frase_pagina_1` es una frase que aparece en la **página impresa número 1** del documento; sirve para calcular la diferencia entre la página del visor de PDF y la página impresa y así citar la página correcta. Si no la conoces, déjala en `None` y se citará la página del visor.

In [ ]:
DOCUMENTOS = [
    {
        "ruta": "FA-CPR-AED-Spanish-Manual.pdf",   # sube el PDF a Colab con este nombre
        "nombre": "Primeros auxilios, RCP y DEA - Manual del participante (4.a ed.)",
        "entidad": "Cruz Roja Americana",
        "frase_pagina_1": "Las emergencias médicas pueden ocurrir todos los días",
    },
]

MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_LINEA_BASE = "sentence-transformers/all-MiniLM-L6-v2"      # el del profesor, solo para comparar
MODELO_GENERADOR  = "Qwen/Qwen2.5-1.5B-Instruct"                  # alternativa mas liviana: "Qwen/Qwen2.5-0.5B-Instruct"
USAR_GENERADOR    = True

PALABRAS_POR_FRAGMENTO = 160   # tamano aproximado de cada fragmento
SOLAPE_PALABRAS        = 30    # cuanto se repite entre fragmentos consecutivos
TOP_K                  = 4     # fragmentos que se entregan al generador
UMBRAL_SIMILITUD       = 0.30  # por debajo: "no hay informacion suficiente" (calibrar en la seccion 8)
USAR_OCR               = False # True si hay paginas con texto ilegible (ver seccion 2)
NUMERO_EMERGENCIAS     = "123" # linea unica de emergencias en Colombia

## 2. Extraer el texto del PDF, página por página (y reparar el texto dañado)

Dos cuidados importantes:

1. Se guarda la página de cada texto desde el principio, porque sin eso no hay trazabilidad.
2. **El manual de la Cruz Roja tiene un problema de codificación de fuentes**: en varias páginas (aproximadamente 1 de cada 6 líneas del texto que revisé) el texto "se extrae" pero sale ilegible, por ejemplo `SXOPRQHV` en lugar de `pulmones`. Si no se corrige, esas páginas, entre ellas gran parte del capítulo de atragantamiento, quedan invisibles para el buscador.

El patrón es regular: cada carácter está desplazado 29 posiciones en la tabla ASCII (el espacio aparece como un carácter de control), y las letras con tilde, `ñ` y las ligaduras `fi`/`fl` usan códigos propios. `reparar_texto` deshace ese desplazamiento **palabra por palabra** y solo acepta la versión reparada cuando es una palabra que ya existe en las partes sanas del propio documento. Así no daña el texto limpio ni otros PDF.

Después se mide la calidad de cada página (proporción de palabras comunes del español). Si aún quedan páginas ilegibles, `USAR_OCR = True` las lee con OCR como último recurso.

In [ ]:
import re, io, json, unicodedata
from collections import Counter
import numpy as np
import pymupdf

PALABRAS_COMUNES = set(
    "de la el que en y a los se del las un por con no una su para es al lo como mas o pero sus le ya este si "
    "porque esta entre cuando muy sin sobre tambien me hasta hay donde quien desde todo nos durante todos uno "
    "les ni contra otros ese eso ante ellos e esto antes algunos unos otro otras otra tanto esa estos mucho "
    "nada muchos cual poco ella estar estas algunas algo persona puede debe".split()
)

def sin_acentos(txt):
    return "".join(c for c in unicodedata.normalize("NFD", txt) if unicodedata.category(c) != "Mn")

def calidad_texto(txt):
    """Proporcion de palabras comunes del espanol. Texto normal: ~0.30-0.45. Texto danado: cerca de 0."""
    tokens = re.findall(r"[a-z]+", sin_acentos(txt).lower())
    if len(tokens) < 30:
        return 1.0   # muy poco texto: no se puede juzgar
    return sum(t in PALABRAS_COMUNES for t in tokens) / len(tokens)

# --- Reparacion de la codificacion danada (especifica del manual de la Cruz Roja) ---
DESPLAZAMIENTO = 29
ESPECIALES = {"i": "á", "p": "é", "t": "í", "y": "ó", "~": "ú", "x": "ñ", "¿": "fi", "À": "fl"}
CONTROL = re.compile(r"[\x01-\x08\x0b-\x1f]")

def decodificar(token):
    salida = []
    for c in token:
        if c in ESPECIALES:
            salida.append(ESPECIALES[c])
        elif 0x03 <= ord(c) <= 0x5d:
            salida.append(chr(ord(c) + DESPLAZAMIENTO))
        else:
            salida.append(c)
    return "".join(salida)

def nucleo(token):
    return re.sub(r"[^a-záéíóúñü]", "", token.lower())

def construir_vocabulario(paginas):
    """Palabras sanas del propio documento: minusculas o con inicial mayuscula, vistas 2 veces o mas."""
    cuenta = Counter()
    for _, txt in paginas:
        for w in re.findall(r"[A-Za-zÁÉÍÓÚÑáéíóúñü]+", txt):
            if re.fullmatch(r"[A-ZÁÉÍÓÚÑ]?[a-záéíóúñü]{2,}", w):
                cuenta[w.lower()] += 1
    return {w for w, n in cuenta.items() if n >= 2 and len(w) >= 3}

def reparar_texto(txt, vocab):
    partes = re.split(r"([\s\x03]+)", txt)
    salida = []
    for p in partes:
        if not p:
            continue
        if re.fullmatch(r"[\s\x03]+", p):
            salida.append(" " if "\x03" in p else p)
            continue
        if p == "\\":
            salida.append("y"); continue
        cand = decodificar(p)
        if nucleo(p) in vocab:
            salida.append(p)                       # ya es una palabra conocida
        elif nucleo(cand) in vocab:
            salida.append(cand)                    # la version decodificada si lo es
        elif CONTROL.search(p) or "¿" in p or "À" in p:
            salida.append(cand)                    # rastro claro de la codificacion danada
        else:
            salida.append(p)
    return "".join(salida)

def ocr_pagina(pagina, dpi=200):
    import pytesseract
    from PIL import Image
    pix = pagina.get_pixmap(dpi=dpi)
    return pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))), lang="spa")

def extraer_paginas(ruta, usar_ocr=False, umbral_calidad=0.15):
    with pymupdf.open(ruta) as pdf:
        crudas = [(i, pag.get_text("text")) for i, pag in enumerate(pdf, start=1)]
        vocab = construir_vocabulario(crudas)
        paginas = [(i, reparar_texto(txt, vocab)) for i, txt in crudas]
        antes = sum(calidad_texto(t) < umbral_calidad for _, t in crudas)
        sospechosas = [i for i, t in paginas if calidad_texto(t) < umbral_calidad]
        if usar_ocr:
            for i in sospechosas:
                paginas[i - 1] = (i, ocr_pagina(pdf[i - 1]))
    return paginas, sospechosas, antes

In [ ]:
paginas_por_doc = {}
for doc in DOCUMENTOS:
    paginas, sospechosas, antes = extraer_paginas(doc["ruta"], usar_ocr=USAR_OCR)
    paginas_por_doc[doc["nombre"]] = paginas
    print(f"{doc['nombre']}: {len(paginas)} paginas")
    print(f"  paginas ilegibles antes de reparar: {antes} | despues: {len(sospechosas)}")
    if sospechosas and not USAR_OCR:
        print(f"  Aun ilegibles: {sospechosas}")
        print("  Prueba USAR_OCR = True (y descomenta la linea de OCR en la celda de instalacion).")

In [ ]:
# Revisa a ojo una pagina que estaba danada: debe leerse como espanol normal
pagina_muestra = 66 if len(paginas_por_doc[DOCUMENTOS[0]["nombre"]]) > 66 else 1
print(paginas_por_doc[DOCUMENTOS[0]["nombre"]][pagina_muestra - 1][1][:900])

## 3. Limpiar y dividir en fragmentos con metadatos

- Se eliminan encabezados y números de página que se repiten (ruido que confunde la búsqueda).
- Cada fragmento se arma con oraciones completas, de unas 160 palabras, con un pequeño solape con el anterior.
- Un fragmento nunca cruza de una página a otra, así la cita de página es exacta.
- Cada fragmento guarda: identificador, texto, documento, entidad, página del visor, página impresa y tema. Es la tabla "información útil de la base" del planteamiento.

In [ ]:
RUIDO = re.compile(
    r"^[ \t]*(\d+[ \t]+)?Primeros auxilios[,/ ].*(Manual (del|para el) participante|Acerca del presente manual|Precauciones).*$"
    r"|^[ \t]*CAP[IÍ]TULO \d+ \|.*$"
    r"|^[ \t]*\d{1,3}[ \t]*$",
    re.IGNORECASE | re.MULTILINE,
)

def limpiar(txt):
    txt = re.sub(r"[\x00-\x08\x0b-\x1f\xa0]", " ", txt)   # restos de control y espacios duros
    txt = RUIDO.sub(" ", txt)
    txt = re.sub(r"-\s*\n\s*", "", txt)          # palabras cortadas con guion al final de linea
    txt = txt.replace("\u00ad", "")
    return re.sub(r"\s+", " ", txt).strip()

TEMAS = {
    "RCP": ["rcp", "compresiones", "reanimaci"],
    "DEA": ["dea ", "desfibril"],
    "atragantamiento": ["atragant"],
    "sangrado y heridas": ["sangrado", "hemorragia", "torniquete", "apósito", "herida"],
    "quemaduras": ["quemadura"],
    "estado de shock": ["shock"],
    "convulsiones": ["convulsi", "epilepsia"],
    "desmayo": ["desmay"],
    "derrame cerebral": ["derrame cerebral"],
    "asma": ["asma", "inhalador"],
    "anafilaxia y alergias": ["anafilaxia", "epinefrina", "alérg"],
    "calor y frío": ["hipotermia", "golpe de calor", "congelamiento", "calambres por calor", "agotamiento por calor"],
    "mordeduras y picaduras": ["picadura", "mordedura", "garrapata", "serpiente"],
    "fracturas y lesiones": ["fractura", "esguince", "dislocaci", "tablilla", "lesión en la cabeza"],
    "envenenamiento": ["envenen", "veneno"],
    "llamar a emergencias": ["9-1-1", "número de emergencia"],
}

def asignar_tema(txt):
    bajo = txt.lower()
    puntajes = {t: sum(bajo.count(p) for p in palabras) for t, palabras in TEMAS.items()}
    mejor = max(puntajes, key=puntajes.get)
    return mejor if puntajes[mejor] > 0 else "general"

def dividir_en_fragmentos(texto, palabras_max, solape):
    oraciones = re.split(r"(?<=[.!?:;])\s+", texto)
    fragmentos, actual, n = [], [], 0
    for o in oraciones:
        w = len(o.split())
        if actual and n + w > palabras_max:
            fragmentos.append(" ".join(actual))
            # conservar al final unas pocas palabras como solape
            cola, m = [], 0
            for prev in reversed(actual):
                pw = len(prev.split())
                if m + pw > solape:
                    break
                cola.insert(0, prev); m += pw
            actual, n = cola, m
        actual.append(o); n += w
    if actual:
        fragmentos.append(" ".join(actual))
    return fragmentos

def calcular_desfase(paginas, frase):
    """Pagina del visor donde aparece la pagina impresa 1 -> desfase = pagina_visor - 1."""
    if frase:
        for i, txt in paginas:
            if frase.lower() in limpiar(txt).lower():
                return i - 1
    return 0

def construir_fragmentos(documentos, paginas_por_doc):
    chunks = []
    for doc in documentos:
        paginas = paginas_por_doc[doc["nombre"]]
        desfase = calcular_desfase(paginas, doc.get("frase_pagina_1"))
        for num_visor, txt in paginas:
            limpio = limpiar(txt)
            if len(limpio) < 80:
                continue
            for pedazo in dividir_en_fragmentos(limpio, PALABRAS_POR_FRAGMENTO, SOLAPE_PALABRAS):
                if len(pedazo.split()) < 15:
                    continue
                chunks.append({
                    "id": f"c{len(chunks):05d}",
                    "texto": pedazo,
                    "documento": doc["nombre"],
                    "entidad": doc["entidad"],
                    "pagina_visor": num_visor,
                    "pagina_impresa": num_visor - desfase,
                    "tema": asignar_tema(pedazo),
                })
    return chunks

In [ ]:
chunks = construir_fragmentos(DOCUMENTOS, paginas_por_doc)
print(f"{len(chunks)} fragmentos")
for c in chunks[100:103]:
    print(f"\n[{c['id']}] {c['entidad']} | p. {c['pagina_impresa']} (visor {c['pagina_visor']}) | tema: {c['tema']}")
    print(c["texto"][:300], "...")

with open("fragmentos.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=1)

## 4. Índices de búsqueda

Se construyen dos índices sobre los mismos fragmentos:

- **Semántico**: embeddings multilingües normalizados, de modo que el producto punto equivale a la similitud coseno. Entiende que "se atraganta" y "obstrucción de la vía respiratoria" hablan de lo mismo.
- **De palabras clave (TF-IDF)**: encuentra términos exactos como "9-1-1", "DEA" o "torniquete" que a veces los embeddings diluyen. Se recortan las palabras a sus primeras 6 letras (un "stemming" casero) para que *atraganta* y *atragantamiento* coincidan.

En la búsqueda híbrida ambos rankings se combinan con *reciprocal rank fusion*, que no necesita calibrar las escalas de los dos puntajes.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

class IndiceSemantico:
    def __init__(self, nombre_modelo, chunks):
        self.modelo = SentenceTransformer(nombre_modelo)
        self.matriz = self.modelo.encode([c["texto"] for c in chunks],
                                         normalize_embeddings=True, batch_size=32, show_progress_bar=True)
    def similitudes(self, consulta):
        q = self.modelo.encode([consulta], normalize_embeddings=True)[0]
        return self.matriz @ q

STOP_SIN_ACENTOS = {sin_acentos(p) for p in PALABRAS_COMUNES}

def analizador(txt):
    tokens = re.findall(r"[a-z0-9]+", sin_acentos(txt).lower())
    return [t[:6] for t in tokens if t not in STOP_SIN_ACENTOS]

class IndiceLexico:
    def __init__(self, chunks):
        self.vec = TfidfVectorizer(analyzer=analizador, sublinear_tf=True)
        self.matriz = self.vec.fit_transform([c["texto"] for c in chunks])
    def puntajes(self, consulta):
        return (self.matriz @ self.vec.transform([consulta]).T).toarray().ravel()

In [ ]:
indice_sem = IndiceSemantico(MODELO_EMBEDDINGS, chunks)
indice_lex = IndiceLexico(chunks)
print("Indices listos:", indice_sem.matriz.shape)

## 5. Recuperación

`buscar` devuelve los `k` mejores fragmentos y un indicador de si la información parece suficiente. Ese indicador se calcula con la **similitud semántica máxima** entre la pregunta y cualquier fragmento: si ni el mejor fragmento se parece lo bastante, el sistema prefiere decir que no sabe.

`expandir_consulta` es la mejora de "expansión de consulta" del planteamiento, en su versión más simple: un diccionario que agrega a la pregunta el vocabulario técnico del manual cuando el usuario usa palabras cotidianas. El grupo puede ampliarlo con los términos que vea fallar en la evaluación.

In [ ]:
SINONIMOS = {
    "atraga": "atragantamiento obstrucción de la vía respiratoria golpes en la espalda presiones abdominales",
    "ahog": "atragantamiento ahogamiento vía respiratoria",
    "se ahoga": "atragantamiento no puede toser hablar ni respirar",
    "desmay": "desmayo pérdida de conciencia persona inconsciente",
    "desvane": "desmayo pérdida de conciencia",
    "sangr": "sangrado hemorragia presión directa apósito",
    "hemorrag": "sangrado hemorragia presión directa",
    "quem": "quemadura enfriar con agua corriente",
    "infarto": "ataque al corazón emergencia cardíaca dolor en el pecho",
    "paro": "paro cardíaco RCP compresiones en el pecho",
    "no respira": "paro respiratorio RCP compresiones en el pecho",
    "ataque de epilepsia": "convulsión epilepsia",
    "convuls": "convulsión epilepsia",
    "diente": "lesiones en los dientes",
    "nariz": "hemorragia nasal lesiones en la nariz",
    "hueso roto": "fractura tablilla inmovilizar",
    "fractura": "fractura tablilla inmovilizar",
    "azucar": "diabetes hipoglucemia emergencia diabética",
    "diabet": "diabetes hipoglucemia hiperglucemia",
    "alergi": "reacción alérgica anafilaxia autoinyector de epinefrina",
    "abeja": "picadura de insecto aguijón anafilaxia",
    "garrapata": "garrapata enfermedades transmitidas por garrapatas retirar con pinzas",
    "serpiente": "mordedura de serpiente venenosa",
    "golpe en la cabeza": "lesión en la cabeza conmoción cerebral",
    "frio": "hipotermia congelamiento emergencias relacionadas con el frío",
    "calor": "golpe de calor agotamiento por calor calambres por calor",
    "asma": "ataque de asma inhalador de alivio rápido",
    "derrame": "derrame cerebral FAST rostro brazo habla tiempo",
    "veneno": "envenenamiento centro de control de envenenamiento",
    "intoxic": "envenenamiento centro de control de envenenamiento",
    "ambulancia": "llamar al número de emergencia local",
}

def expandir_consulta(consulta):
    bajo = sin_acentos(consulta.lower())
    extra = [v for k, v in SINONIMOS.items() if sin_acentos(k) in bajo]
    return consulta + (" " + " ".join(extra) if extra else "")

def buscar(consulta, modo="hibrido", k=TOP_K, expandir=False, idx_sem=None, idx_lex=None):
    idx_sem = idx_sem or indice_sem
    idx_lex = idx_lex or indice_lex
    q = expandir_consulta(consulta) if expandir else consulta
    sem = idx_sem.similitudes(q)
    if modo == "semantico":
        orden = np.argsort(-sem)[:k]
        puntaje = {int(i): float(sem[i]) for i in orden}
    else:
        lex = idx_lex.puntajes(q)
        rank_sem = {int(i): r for r, i in enumerate(np.argsort(-sem))}
        rank_lex = {int(i): r for r, i in enumerate(np.argsort(-lex))}
        fusion = {i: 1/(60 + rank_sem[i]) + (1/(60 + rank_lex[i]) if lex[i] > 0 else 0) for i in rank_sem}
        orden = sorted(fusion, key=fusion.get, reverse=True)[:k]
        puntaje = {int(i): fusion[i] for i in orden}
    resultados = [{**chunks[int(i)], "similitud": float(sem[int(i)]), "puntaje": puntaje[int(i)]} for i in orden]
    return resultados, bool(sem.max() >= UMBRAL_SIMILITUD), float(sem.max())

In [ ]:
res, suficiente, maxsim = buscar("¿qué hago si alguien se atraganta?", expandir=True)
print(f"similitud maxima: {maxsim:.3f} | informacion suficiente: {suficiente}")
for r in res:
    print(f"- p. {r['pagina_impresa']} | {r['tema']} | sim {r['similitud']:.3f} | {r['texto'][:110]}...")

## 6. Generación con citas y reglas de seguridad

Tres decisiones de diseño, pensadas para un tema de salud:

1. **Las fuentes las arma el código**, no el modelo. El modelo redacta la respuesta; la lista de documento y página sale directamente de los fragmentos recuperados, así que no puede inventarse una cita.
2. **El aviso de emergencia es texto fijo**. Si la pregunta suena urgente, el sistema antepone el recordatorio de llamar al número de emergencias, sin depender de que el modelo lo recuerde. Esto importa porque el manual es de EE. UU. y habla del 9-1-1; en Colombia el número es el 123.
3. **El prompt obliga a responder solo con los fragmentos** y a reconocer cuando no alcanzan. Un modelo pequeño no siempre obedece, por eso la evaluación manual de "¿está respaldada?" de la sección 8 sigue siendo necesaria.

Con `USAR_GENERADOR = False` se omite el modelo y se muestran los fragmentos más relevantes.

In [ ]:
if USAR_GENERADOR:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(MODELO_GENERADOR)
    generador = AutoModelForCausalLM.from_pretrained(MODELO_GENERADOR, torch_dtype="auto", device_map="auto")

PROMPT_SISTEMA = (
    "Eres un asistente educativo de primeros auxilios. Respondes en español, con frases cortas y claras, "
    "para una persona sin formación en salud. Usa ÚNICAMENTE la información de los FRAGMENTOS que se te dan. "
    "Si los fragmentos no contienen la respuesta, di exactamente: 'No encontré información suficiente en los documentos.' "
    "No diagnostiques, no recetes medicamentos y no inventes datos. "
    "Cuando uses un fragmento, cítalo al final de la frase con su número entre corchetes, por ejemplo [1]. "
    "Si hay pasos, enuméralos en el orden en que aparecen en los fragmentos."
)

def generar_respuesta(consulta, resultados, max_tokens=350):
    contexto = "\n\n".join(f"[{i}] ({r['entidad']}, p. {r['pagina_impresa']}) {r['texto']}" for i, r in enumerate(resultados, 1))
    mensajes = [{"role": "system", "content": PROMPT_SISTEMA},
                {"role": "user", "content": f"FRAGMENTOS:\n{contexto}\n\nPREGUNTA: {consulta}"}]
    texto = tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = tok(texto, return_tensors="pt").to(generador.device)
    salida = generador.generate(**entradas, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(salida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

URGENTE = re.compile(
    r"no respira|inconsciente|atragant|se ahoga|sangr|hemorrag|convuls|quemadur|infarto|paro cardi|desmay|"
    r"envenen|intoxic|anafilax|derrame|serpiente|electrocut|golpe de calor|hipotermia|fractura|ataque",
    re.IGNORECASE)

def es_urgente(consulta):
    return bool(URGENTE.search(sin_acentos(consulta)))

AVISO_EMERGENCIA = (f"AVISO: si hay una persona en peligro, llama de inmediato al {NUMERO_EMERGENCIAS} "
                    "(línea única de emergencias en Colombia; confirma el número de tu localidad). "
                    "El manual fuente es de EE. UU. y menciona el 9-1-1.")
SIN_INFO = ("No encontré información suficiente en los documentos para responder esa pregunta. "
            "Esta herramienta es solo educativa; ante una duda sobre la salud de alguien, consulta a un profesional.")

def preguntar(consulta, modo="hibrido", expandir=True, k=TOP_K, mostrar_fragmentos=False):
    resultados, suficiente, maxsim = buscar(consulta, modo=modo, k=k, expandir=expandir)
    partes = [f"Pregunta: {consulta}\n"]
    if es_urgente(consulta):
        partes.append(AVISO_EMERGENCIA + "\n")
    if not suficiente:
        partes.append(SIN_INFO)
        print("\n".join(partes)); return
    if USAR_GENERADOR:
        partes.append("Respuesta:\n" + generar_respuesta(consulta, resultados))
    else:
        partes.append("Fragmentos más relevantes (sin generador):")
        for i, r in enumerate(resultados[:2], 1):
            partes.append(f"[{i}] {r['texto'][:500]}...")
    partes.append("\nFuentes consultadas:")
    for i, r in enumerate(resultados, 1):
        partes.append(f"  [{i}] {r['entidad']} - {r['documento']}, p. {r['pagina_impresa']} (tema: {r['tema']}, similitud {r['similitud']:.2f})")
    if mostrar_fragmentos:
        partes.append("\nTexto de los fragmentos:")
        partes += [f"  [{i}] {r['texto']}" for i, r in enumerate(resultados, 1)]
    print("\n".join(partes))

In [ ]:
preguntar("¿qué hago si un adulto se está atragantando y está consciente?")

In [ ]:
preguntar("¿cómo se prepara una torta de chocolate?")   # fuera del corpus: debe decir que no sabe

## 7. Preguntas de prueba

Cada pregunta está escrita como la haría un usuario cotidiano y lleva las **páginas impresas** del manual donde está la respuesta. Se considera acierto si algún fragmento recuperado cae en esas páginas o a una de distancia (la información a veces continúa en la página siguiente). Las últimas preguntas están fuera del corpus a propósito: el sistema debería rechazarlas.

Estas páginas esperadas las anoté leyendo el manual; **el grupo debe verificarlas** y ampliar la lista a 30 preguntas o más, idealmente con preguntas escritas por personas ajenas al equipo.

In [ ]:
PREGUNTAS_PRUEBA = [
    ("¿qué hago si un adulto se está atragantando y está consciente?", [61, 62, 65]),
    ("mi bebé se atraganta y no puede llorar, ¿qué hago?", [63, 64, 67]),
    ("¿cómo hago RCP a un bebé?", [39, 44]),
    ("¿cómo se usa el desfibrilador en un niño pequeño?", [48, 53]),
    ("me echaron un químico en la piel, ¿qué hago?", [108]),
    ("alguien sangra muchísimo de una herida, ¿cómo lo detengo?", [105, 114]),
    ("¿cómo sé si alguien está teniendo un derrame cerebral?", [73]),
    ("una persona está convulsionando, ¿qué hago?", [71]),
    ("encontré a alguien helado y temblando en el frío", [84, 85]),
    ("me encontré una garrapata pegada en la piel", [88, 89]),
    ("¿cuándo debo llamar a la línea de emergencias?", [10]),
    ("¿cómo paro una hemorragia de la nariz?", [110]),
    ("mi hijo tiene un ataque de asma, ¿cómo le ayudo con el inhalador?", [151, 152]),
    ("¿cómo uso un autoinyector de epinefrina?", [156, 157, 158]),
    ("¿cuándo se usa un torniquete?", [106, 115]),
    ("una persona se desmayó de repente", [70]),
    ("¿cómo reconozco un golpe de calor?", [83]),
    ("me picó una abeja y me estoy hinchando", [85]),
]
PREGUNTAS_FUERA_DEL_CORPUS = [
    "¿cómo se prepara una torta de chocolate?",
    "¿qué antibiótico debo tomar para la gripe?",
    "¿quién ganó el último mundial de fútbol?",
    "¿cuántas calorías tiene una arepa?",
]

## 8. Evaluación: ¿mejoramos frente al ejemplo del profesor?

Se comparan varias configuraciones sobre las mismas preguntas:

- **Línea base**: lo que hacía `RAG_simple` (modelo en inglés, solo semántica), pero aplicado a los fragmentos del manual.
- **Semántico multilingüe**, **híbrido** y **híbrido + expansión de consulta**.

La métrica es el *acierto en los primeros k* (hit@k): en qué porcentaje de preguntas aparece una página correcta entre los k primeros fragmentos. Es la forma más directa de medir la variable "relevancia de los fragmentos recuperados" del planteamiento.

In [ ]:
import pandas as pd

def acierta(resultados, paginas_esperadas, tolerancia=1):
    return any(abs(r["pagina_impresa"] - p) <= tolerancia for r in resultados for p in paginas_esperadas)

def evaluar(nombre, **kwargs):
    aciertos = {1: 0, 3: 0, 5: 0}
    fallos = []
    for pregunta, paginas in PREGUNTAS_PRUEBA:
        res, _, _ = buscar(pregunta, k=5, **kwargs)
        for k in aciertos:
            if acierta(res[:k], paginas):
                aciertos[k] += 1
        if not acierta(res[:5], paginas):
            fallos.append(pregunta)
    n = len(PREGUNTAS_PRUEBA)
    return {"configuracion": nombre, **{f"hit@{k}": round(v / n, 2) for k, v in aciertos.items()}}, fallos

indice_base = IndiceSemantico(MODELO_LINEA_BASE, chunks)

filas, fallos_por_config = [], {}
for nombre, kw in [
    ("Línea base (profesor): MiniLM inglés, semántico", dict(modo="semantico", expandir=False, idx_sem=indice_base)),
    ("Semántico multilingüe",                           dict(modo="semantico", expandir=False)),
    ("Híbrido",                                         dict(modo="hibrido", expandir=False)),
    ("Híbrido + expansión de consulta",                 dict(modo="hibrido", expandir=True)),
]:
    fila, fallos = evaluar(nombre, **kw)
    filas.append(fila); fallos_por_config[nombre] = fallos

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# Preguntas que cada configuracion no logro resolver: de aqui salen las mejoras de la siguiente iteracion
for nombre, fallos in fallos_por_config.items():
    print(f"\n{nombre}: {len(fallos)} fallos")
    for f in fallos:
        print("  -", f)

### Calibrar el umbral de "no hay información suficiente"

El valor `0.30` es un punto de partida. Cada modelo de embeddings tiene su propia escala, así que hay que mirar cómo se separan las preguntas que sí están en el manual de las que no.

In [ ]:
dentro = [buscar(p, k=1)[2] for p, _ in PREGUNTAS_PRUEBA]
fuera  = [buscar(p, k=1)[2] for p in PREGUNTAS_FUERA_DEL_CORPUS]
print(f"Similitud maxima, preguntas DENTRO del corpus: min {min(dentro):.2f} | promedio {np.mean(dentro):.2f}")
print(f"Similitud maxima, preguntas FUERA del corpus:  max {max(fuera):.2f} | promedio {np.mean(fuera):.2f}")
if min(dentro) > max(fuera):
    print(f"Umbral sugerido: entre {max(fuera):.2f} y {min(dentro):.2f} (por ejemplo {(max(fuera)+min(dentro))/2:.2f})")
else:
    print("Las dos distribuciones se solapan: el umbral solo no basta; conviene mas preguntas de prueba o un reordenador (reranker).")
rechazadas = sum(s < UMBRAL_SIMILITUD for s in fuera)
print(f"Con UMBRAL_SIMILITUD = {UMBRAL_SIMILITUD}: se rechazan {rechazadas}/{len(fuera)} fuera del corpus y "
      f"se pierden {sum(s < UMBRAL_SIMILITUD for s in dentro)}/{len(dentro)} dentro del corpus.")

## 9. Qué sigue (siguientes iteraciones)

- Revisar a mano, para 10 a 15 preguntas, si la respuesta generada **está respaldada** por los fragmentos y si **responde** lo preguntado; anotar los casos donde inventa (esa es la segunda parte de la variable a analizar).
- Agregar las fuentes abiertas del planteamiento (ICBF, CAR, MedlinePlus) en `DOCUMENTOS` y repetir la evaluación; comprobar si el acierto sube o si aparecen fragmentos contradictorios.
- Probar un reordenador (reranker) multilingüe sobre los 10 primeros fragmentos.
- Si la materia lo exige, exponer `preguntar` como servicio (por ejemplo, un webhook que consuma n8n o Flowise) o con una interfaz sencilla.